In [1]:
import pandas as pd

import re

from tqdm.notebook import tqdm
tqdm.pandas()

In [2]:
df = pd.read_excel("LLM_outputs.xlsx")

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10584 entries, 0 to 10583
Data columns (total 14 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   Unnamed: 0                     10584 non-null  int64 
 1   id                             10584 non-null  int64 
 2   company                        10584 non-null  object
 3   job title                      10584 non-null  object
 4   text                           10582 non-null  object
 5   triples_qwen_structured        10584 non-null  object
 6   triples_qwen_semi-structured   10584 non-null  object
 7   triples_qwen_unstructured      10584 non-null  object
 8   triples_gemma_structured       10584 non-null  object
 9   triples_gemma_semi-structured  10584 non-null  object
 10  triples_gemma_unstructured     10584 non-null  object
 11  triples_llama_structured       10584 non-null  object
 12  triples_llama_semi-structured  10584 non-null  object
 13  t

In [4]:
import re

def extract_triples(row, col_name):
    """
    row: the dataframe row
    col_name: the current column being processed
    """
    raw_text = row[col_name]
    if not raw_text or not isinstance(raw_text, str):
        return []

    guaranteed_list = [
        "REQUIRES_SKILL", "REQUIRES_QUALITY", "DESIRES", "REQUIRES_EXPERIENCE_LEVEL",
        "REQUIRES_WORK_EXPERIENCE", "REQUIRES_EDUCATION", "REQUIRES_CERTIFICATION",
        "REQUIRES_LANGUAGE", "INVOLVES_TASK", "OFFERS_POSITION", "HAS_LOCATION",
        "HAS_CONTRACT_TYPE", "HAS_SALARY_RANGE", "IS_MANAGER_LEVEL", "IS_IN_INDUSTRY"
    ]

    clean_text = raw_text.replace('\n', ',').replace("```python", "").replace("`", "").replace('""', '"')
    
    # We only force-split if the column name implies strict schema enforcement.
    if "structured" in col_name.lower() and "unstructured" not in col_name.lower():
        pattern = r'\b(' + '|'.join(guaranteed_list) + r')\b'
        # re.split keeps the delimiter if we use parentheses; we join with commas to preserve your logic
        parts = re.split(pattern, clean_text)
        clean_text = ",".join(parts)

    # Strip all quotes
    clean_text = clean_text.replace('"', '').replace("'", "")
    # Standardize all delimiters to commas
    clean_text = clean_text.replace('[', ',').replace(']', ',').replace('(', ',').replace(')', ',')

    # Tokenization by comma
    raw_parts = clean_text.split(',')
    tokens = []
    for part in raw_parts:
        part = part.strip()
        if not part: continue
        
        # Strip "label:" noise
        if ':' in part and not part.startswith('http'):
            sub_parts = part.split(':', 1)
            if len(sub_parts) == 2 and ' ' not in sub_parts[0]:
                part = sub_parts[1]
        tokens.append(part.strip())

    # --- ALIGNMENT POST-PROCESS ---
    while len(tokens) >= 3:
        s_low = tokens[0].lower()
        p_low = tokens[1].lower()
        
        is_filler = any(f in s_low for f in ["heres a", "here are", "the triples", "list of", "comprehensive", "entity relations"])
        
        # Branching logic for alignment
        if "unstructured" in col_name.lower():
             # For unstructured, we trust your original alignment skip
             if is_filler:
                 tokens.pop(0)
                 continue
             break
        else:
            # For structured/semi, we also check if the predicate looks like junk (has spaces)
            if is_filler or (" " in p_low and p_low.upper() not in guaranteed_list):
                tokens.pop(0)
            else:
                break

    # --- GROUPING & PLACEHOLDERS ---
    triples = []
    job_title = str(row['job title']).strip()
    company_name = str(row['company']).strip()
    placeholders = {
        "job": job_title, "job title": job_title, "job listing": job_title,
        "job_title": job_title, "job_listing": job_title, 
        "company": company_name, "company name": company_name, "company_name": company_name
    }

    for i in range(0, len(tokens) - (len(tokens) % 3), 3):
        s, p, o = tokens[i], tokens[i+1], tokens[i+2]
        
        s_final = placeholders.get(s.lower(), s)
        o_final = placeholders.get(o.lower(), o)
        
        if s_final and p and o_final:
            triples.append((s_final, p, o_final))

    return list(dict.fromkeys(triples))

In [5]:
df_clean = df.copy()

for model in ["qwen", "gemma", "llama"]:
    for prompt in ["structured", "semi-structured", "unstructured"]:
        
        df_clean[f"triples_{model}_{prompt}"] = df[["company", "job title", f"triples_{model}_{prompt}"]].progress_apply(
            lambda x:
            extract_triples(x, f"triples_{model}_{prompt}"), 
            axis=1
        )

  0%|          | 0/10584 [00:00<?, ?it/s]

  0%|          | 0/10584 [00:00<?, ?it/s]

  0%|          | 0/10584 [00:00<?, ?it/s]

  0%|          | 0/10584 [00:00<?, ?it/s]

  0%|          | 0/10584 [00:00<?, ?it/s]

  0%|          | 0/10584 [00:00<?, ?it/s]

  0%|          | 0/10584 [00:00<?, ?it/s]

  0%|          | 0/10584 [00:00<?, ?it/s]

  0%|          | 0/10584 [00:00<?, ?it/s]

In [6]:
df_clean.head()

,Unnamed: 0,id,company,job title,text,triples_qwen_structured,triples_qwen_semi-structured,triples_qwen_unstructured,triples_gemma_structured,triples_gemma_semi-structured,triples_gemma_unstructured,triples_llama_structured,triples_llama_semi-structured,triples_llama_unstructured
0,0,1527392,Jobindex,"IT-administrator – få indflydelse på et setup,...",Vil du ind i en virksomhed i rivende udvikling...,"[(IT-administrator, REQUIRES_SKILL, setup mana...","[(IT-administrator, REQUIRES_SKILL, network ma...","[(IT-administrator, requires, technical_setup_...","[(IT-administrator, INVOLVES_TASK, Maintaining...","[(IT-administrator, REQUIRES_SKILL, System Adm...",[(IT-administrator – få indflydelse på et setu...,"[(IT-administrator, REQUIRES_SKILL, setup), (I...","[(REQUIRES_SKILL, IT-administrator, Docker), (...","[(IT-administrator, har, få indflydelse på et ..."
1,1,1527395,Jobindex,"IT-administrator – få indflydelse på et setup,...","IT-administrator – få indflydelse på et setup,...","[(IT-administrator, REQUIRES_SKILL, setup mana...","[(IT-administrator, REQUIRES_SKILL, network ma...","[(IT-administrator, requires, technical_setup_...","[(IT-administrator, REQUIRES_SKILL, IT Adminis...","[(IT-administrator, REQUIRES_SKILL, System adm...",[(IT-administrator – få indflydelse på et setu...,"[(IT-administrator, REQUIRES_SKILL, Docker), (...",[(IT-administrator – få indflydelse på et setu...,"[(IT-administrator, har, få indflydelse på et ..."
2,2,1527397,Aqua d'Or Mineral Water A/S,SQE Manager,For jobsøgere For arbejdsgivere Aqua d'Or Mi...,"[(SQE Manager, REQUIRES_SKILL, Root Cause Anal...","[(SQE Manager, REQUIRES_SKILL, Quality Managem...","[(SQE Manager, requires, quality assurance), (...","[(SQE Manager, REQUIRES_SKILL, Quality Assuran...","[(SQE Manager, REQUIRES_SKILL, Quality Assuran...","[(SQE Manager, title, SQE Manager), (SQE Manag...","[(SQE Manager, REQUIRES_QUALITY, detail-orient...","[(SQE Manager, REQUIRES_SKILL, DevOps Engineer...","[(SQE Manager, requires, English language prof..."
3,3,1527417,Klimabrands,Kundeservice / teknisk support,For jobsøgere For arbejdsgivere mailto:job@k...,"[(Kundeservice / teknisk support, REQUIRES_SKI...","[(Kundeservice / teknisk support, REQUIRES_SKI...","[(Kundeservice / teknisk support, requires, cu...","[(Kundeservice / teknisk support, REQUIRES_SKI...","[(Kundeservice / teknisk support, REQUIRES_SKI...","[(Kundeservice / teknisk support, title, Kunde...","[(Kundeservice, REQUIRES_QUALITY, detail-orien...","[(Kundeservice, REQUIRES_SKILL, Teknisk suppor...","[(Kundeservice, har, en teknisk support specia..."
4,4,1527440,Scan Studio ApS,Retail designer med teknikken på plads,For jobsøgere For arbejdsgivere Scan Studio ...,"[(Retail Designer, REQUIRES_SKILL, Technical D...","[(Retail Designer, REQUIRES_SKILL, Design), (R...","[(Retail Designer, requires, technical skills)...","[(Retail designer, REQUIRES_SKILL, teknikken),...","[(Retail designer, REQUIRES_SKILL, teknikken),...","[(Retail designer med teknikken på plads, has_...","[(Retail designer med teknikken på plads, REQU...","[(Dansk designer, SKILL, REQUIRES_QUALITY), (D...",[(Retail designer requires technisk kompetence...


In [8]:
llm_cols = [c for c in df.columns if 'triples' in c]

empty_percentages = (df_clean[llm_cols].map(lambda x: len(x) == 0).mean() * 100)

# Display the results
print("Percentage of Empty Triple Lists per Column:")
print(empty_percentages.map("{:.2f}%".format))

Percentage of Empty Triple Lists per Column:
triples_qwen_structured          0.12%
triples_qwen_semi-structured     0.01%
triples_qwen_unstructured        0.00%
triples_gemma_structured         0.00%
triples_gemma_semi-structured    0.00%
triples_gemma_unstructured       0.00%
triples_llama_structured         0.01%
triples_llama_semi-structured    0.01%
triples_llama_unstructured       0.83%
dtype: object


In [9]:
df_clean.to_excel("triples_clean.xlsx")